In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)

print("Loading data from folder...")

try:
    df_pbp = pd.read_csv("play_by_play_2025.csv", low_memory=False)
    df_roster = pd.read_csv("roster_2025.csv", low_memory=False)
    df_games = pd.read_csv("games.csv", low_memory=False)
    df_stats_player = pd.read_csv("stats_player_regpost_2025.csv", low_memory=False)
    df_teams = pd.read_csv("teams_colors_logos.csv", low_memory=False)
    
    # Get games only in 2025 season
    df_games = df_games[df_games['season'] == 2025]
    
    print("Data successfully loaded!")
except FileNotFoundError as e:
    print(f"Error: File not found. \nStack: {e}")

print(f"Play-by-play (Plays): {df_pbp.shape[0]} rows, {df_pbp.shape[1]} columns")
print(f"Rosters (Players): {df_roster.shape[0]} rows, {df_roster.shape[1]} columns")
print(f"Games (Games): {df_games.shape[0]} rows, {df_games.shape[1]} columns")
print(f"Player stats: {df_stats_player.shape[0]} rows, {df_stats_player.shape[1]} columns")
print(f"Teams: {df_teams.shape[0]} rows, {df_teams.shape[1]} columns")

# Column cleaning

# Play by play
pbp_cols = ['game_id', 'play_id', 'play_type', 'posteam', 'defteam', 'down', 'ydstogo', 
            'yards_gained', 'passer_player_id', 'receiver_player_id', 'rusher_player_id']
df_pbp_clean = df_pbp[pbp_cols].copy()

print("--- Empty values in dataset columns (Play-by-Play) ---")
print(df_pbp_clean.isna().sum())

# set 0 yard when empty yards columns
df_pbp_clean['yards_gained'] = df_pbp_clean['yards_gained'].fillna(0)
df_pbp_clean['ydstogo'] = df_pbp_clean['ydstogo'].fillna(0)

# Removing plays that doesnt have type (commercials...)
df_pbp_clean = df_pbp_clean.dropna(subset=['play_type'])

print("--- Empty values in dataset columns after cleaning (Play-by-Play) ---")
print(df_pbp_clean.isna().sum())

print("\n--- Preview of cleaned data (Play-by-Play) ---")
display(df_pbp_clean.head(3))

# Roster
roster_cols = ['gsis_id', 'team', 'position', 'full_name', 'height', 'weight', 'college', 'entry_year', 'rookie_year']
df_roster_clean = df_roster[roster_cols].copy()

print("--- Empty values in dataset columns (Roster) ---")
print(df_roster_clean.isna().sum())

print("\n--- Preview of cleaned data (Roster) ---")
display(df_roster_clean.head(3))


#Player stats
stats_cols = ['player_id', 'player_display_name', 'position', 'position_group', 'recent_team', 'games', 
              'passing_yards', 'passing_tds', 'rushing_yards', 'receiving_yards', 'fantasy_points', 'sacks_suffered', 'sack_fumbles', 'sack_fumbles_lost', 'passing_first_downs']
df_stats_player_clean = df_stats_player[stats_cols].copy()

print("--- Empty values in dataset columns (Player stats) ---")
print(df_stats_player_clean.isna().sum())

print("\n--- Preview of cleaned data (Player stats) ---")
display(df_stats_player_clean.head(3))

# Games
games_cols = ['game_id', 'season', 'week', 'game_type', 'home_team', 
              'away_team', 'home_score', 'away_score', 'roof']
df_games_clean = df_games[games_cols].copy()

print("--- Empty values in dataset columns (Games) ---")
print(df_games_clean.isna().sum())

print("\n--- Preview of cleaned data (Games) ---")
display(df_games_clean.head(3))

#Teams
teams_cols = [
    'team_abbr',
    'team_name',
    'team_id',
    'team_conf',
    'team_division'
]

df_teams_clean = df_teams[teams_cols].copy()

print("--- Empty values in dataset columns (Teams) ---")
print(df_teams_clean.isna().sum())

print("\n--- Preview of cleaned data (Teams) ---")
display(df_teams_clean.head(3))


#Statistics
print("\n--- Generating statistics ---")
offensive_plays = ["run", "pass"]
df_pbp_offensive = df_pbp_clean[df_pbp_clean['play_type'].isin(offensive_plays)]

action_statistics = df_pbp_offensive.groupby('play_type')['yards_gained'].agg(['count', 'sum', 'mean', 'max'])
action_statistics.columns = ['Počet akcí', 'Celkem yardů', 'Průměr na akci', 'Nejdelší akce']
print("--- Basic statistics by play type ---")
display(action_statistics.round(2).sort_values('Počet akcí', ascending=False))

top_passers = df_stats_player_clean.sort_values(by='passing_yards', ascending=False).head(5)

top_passers = top_passers.rename(columns={
    'player_display_name': 'Jméno', 
    'recent_team': 'Tým', 
    'passing_yards': 'Passing yards', 
    'passing_tds': 'Passing TDs'
})

print("\n--- TOP 5 QBs (Passing yards 2025) ---")
display(top_passers[['Jméno', 'Tým', 'Passing yards', 'Passing TDs']])

Loading data from folder...
Data successfully loaded!
Play-by-play (Plays): 48771 rows, 372 columns
Rosters (Players): 3137 rows, 36 columns
Games (Games): 285 rows, 46 columns
Player stats: 2025 rows, 113 columns
Teams: 36 rows, 16 columns
--- Empty values in dataset columns (Play-by-Play) ---
game_id                   0
play_id                   0
play_type              1455
posteam                2729
defteam                2729
down                   7993
ydstogo                   0
yards_gained           1511
passer_player_id      28952
receiver_player_id    31189
rusher_player_id      33424
dtype: int64
--- Empty values in dataset columns after cleaning (Play-by-Play) ---
game_id                   0
play_id                   0
play_type                 0
posteam                2140
defteam                2140
down                   6548
ydstogo                   0
yards_gained              0
passer_player_id      27497
receiver_player_id    29734
rusher_player_id      31969
dtype

,game_id,play_id,play_type,posteam,defteam,down,ydstogo,yards_gained,passer_player_id,receiver_player_id,rusher_player_id
1,2025_01_ARI_NO,40,kickoff,ARI,NO,NaN,0,0.0,NaN,NaN,NaN
2,2025_01_ARI_NO,63,run,ARI,NO,1.0,10,3.0,NaN,NaN,00-0033553
3,2025_01_ARI_NO,85,pass,ARI,NO,2.0,7,11.0,00-0035228,00-0037744,NaN


--- Empty values in dataset columns (Roster) ---
gsis_id          4
team             0
position         0
full_name        0
height         163
weight           0
college        182
entry_year       0
rookie_year      0
dtype: int64

--- Preview of cleaned data (Roster) ---


,gsis_id,team,position,full_name,height,weight,college,entry_year,rookie_year
0,NaN,GB,DL,Dante Barnett,NaN,275,NaN,2025,2025
1,00-0022942,IND,QB,Philip Rivers,77.0,228,N.C. State,2004,2004
2,00-0023459,PIT,QB,Aaron Rodgers,74.0,225,California; Butte College,2005,2005


--- Empty values in dataset columns (Player stats) ---
player_id              1
player_display_name    1
position               1
position_group         1
recent_team            0
games                  0
passing_yards          0
passing_tds            0
rushing_yards          0
receiving_yards        0
fantasy_points         0
sacks_suffered         0
sack_fumbles           0
sack_fumbles_lost      0
passing_first_downs    0
dtype: int64

--- Preview of cleaned data (Player stats) ---


...

,team_abbr,team_name,team_id,team_conf,team_division
0,ARI,Arizona Cardinals,3800,NFC,NFC West
1,ATL,Atlanta Falcons,200,NFC,NFC South
2,BAL,Baltimore Ravens,325,AFC,AFC North



--- Generating statistics ---
--- Basic statistics by play type ---


,Počet akcí,Celkem yardů,Průměr na akci,Nejdelší akce
play_type,,,,
pass,19735,119889.0,6.07,88.0
run,14893,67012.0,4.50,93.0



--- TOP 5 QBs (Passing yards 2025) ---


,Jméno,Tým,Passing yards,Passing TDs
8,Matthew Stafford,LA,5643,52
1657,Drake Maye,NE,5222,37
417,Sam Darnold,SEA,4720,30
170,Jared Goff,DET,4564,34
1703,Caleb Williams,CHI,4560,31
